# Example Hypgeom Family Surface

Canonical hypergeometric notebook using the direct routed API, family-owned padded kernels, and the current hardening diagnostics artifacts.

## Scope

This notebook is the canonical example surface for `example_hypgeom_family_surface`. It runs against the repo source tree through `/src`, shows direct public API usage, summarizes validation and benchmark status, and includes visual summaries.

In [ ]:
import io
import json
import os
import re
import subprocess
import sys
import textwrap
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for p in [cur, *cur.parents]:
        if (p / 'pyproject.toml').exists() and (p / 'src' / 'arbplusjax').exists():
            return p
    raise RuntimeError(f'Could not locate repo root from: {start}')

REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))
os.chdir(REPO_ROOT)

PYTHON = os.getenv('ARBPLUSJAX_PYTHON', sys.executable)
JAX_MODE = os.getenv('JAX_MODE', 'cpu').strip().lower()
JAX_DTYPE = os.getenv('JAX_DTYPE', 'float64').strip().lower()
RUN_ENV = os.environ.copy()
RUN_ENV['PYTHONPATH'] = str(REPO_ROOT / 'src') + os.pathsep + RUN_ENV.get('PYTHONPATH', '')
if JAX_MODE == 'cpu':
    RUN_ENV['JAX_PLATFORMS'] = 'cpu'
elif JAX_MODE == 'gpu':
    RUN_ENV['JAX_PLATFORMS'] = 'cuda'
RUN_ENV['JAX_ENABLE_X64'] = '1' if JAX_DTYPE == 'float64' else '0'
EXAMPLE_INPUT_ROOT = REPO_ROOT / 'examples' / 'inputs' / 'example_hypgeom_family_surface'
EXAMPLE_OUTPUT_ROOT = REPO_ROOT / 'examples' / 'outputs' / 'example_hypgeom_family_surface'
EXAMPLE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

def run(cmd: list[str], *, capture: bool = False):
    print('[cmd]', ' '.join(cmd))
    return subprocess.run(cmd, cwd=REPO_ROOT, env=RUN_ENV, text=True, capture_output=capture, check=True)


## Environment

The notebook reports interpreter, selected JAX mode, and the active backend/device view. Canonical retained execution in this repo state is CPU-oriented, but the notebook calling pattern remains CPU/GPU portable and explicitly parameterized for `float32` and `float64`.

In [ ]:
SUPPORTED_JAX_MODES = ('cpu', 'gpu')
SUPPORTED_JAX_DTYPES = ('float32', 'float64')
if JAX_MODE not in SUPPORTED_JAX_MODES:
    raise ValueError(f'Unsupported JAX_MODE: {JAX_MODE}')
if JAX_DTYPE not in SUPPORTED_JAX_DTYPES:
    raise ValueError(f'Unsupported JAX_DTYPE: {JAX_DTYPE}')
print('python:', PYTHON)
print('jax_mode:', JAX_MODE)
print('jax_dtype:', JAX_DTYPE)
print('supported_jax_modes:', SUPPORTED_JAX_MODES)
print('supported_jax_dtypes:', SUPPORTED_JAX_DTYPES)
print('validation_slice:', 'cpu_current__gpu_portable_contract')
runtime = run([PYTHON, 'tools/check_jax_runtime.py'], capture=True)
print(runtime.stdout)
runtime_payload = json.loads(runtime.stdout)
(EXAMPLE_OUTPUT_ROOT / f'runtime_{JAX_MODE}.json').write_text(json.dumps(runtime_payload, indent=2) + '\n', encoding='utf-8')

## Direct Usage

Exercise the real point 1f1 surface, the interval-mode routed binder, and the regularized lower/upper incomplete-gamma wrappers.

In [ ]:
import jax.numpy as jnp
from arbplusjax import acb_core, api, double_interval as di, hypgeom_wrappers

a = jnp.asarray([1.0, 1.25, 1.5], dtype=jnp.float64)
b = jnp.asarray([2.0, 2.25, 2.5], dtype=jnp.float64)
z = jnp.asarray([0.2, 0.3, 0.4], dtype=jnp.float64)
point_bound = api.bind_point_batch_jit('hypgeom.arb_hypgeom_1f1', dtype='float64', pad_to=8)
a_iv = di.interval(a, a + 0.01)
b_iv = di.interval(b, b + 0.01)
z_iv = di.interval(z, z + 0.01)
interval_bound = api.bind_interval_batch('hypgeom.arb_hypgeom_1f1', mode='rigorous', dtype='float64', pad_to=8, prec_bits=53)
s = di.interval(jnp.asarray([1.2, 1.25], dtype=jnp.float64), jnp.asarray([1.25, 1.3], dtype=jnp.float64))
x = di.interval(jnp.asarray([0.3, 0.35], dtype=jnp.float64), jnp.asarray([0.35, 0.4], dtype=jnp.float64))
sc = acb_core.acb_box(di.interval(jnp.float64(1.2), jnp.float64(1.25)), di.interval(jnp.float64(-0.05), jnp.float64(0.05)))
zc = acb_core.acb_box(di.interval(jnp.float64(0.3), jnp.float64(0.35)), di.interval(jnp.float64(-0.02), jnp.float64(0.02)))
direct_results = {
    'onef1_point': point_bound(a, b, z),
    'onef1_rigorous': interval_bound(a_iv, b_iv, z_iv),
    'gamma_lower_regularized': hypgeom_wrappers.arb_hypgeom_gamma_lower_batch_mode_padded(s, x, pad_to=4, impl='rigorous', prec_bits=53, regularized=True),
    'gamma_upper_regularized': hypgeom_wrappers.arb_hypgeom_gamma_upper_batch_mode_padded(s, x, pad_to=4, impl='adaptive', prec_bits=53, regularized=True),
    'complex_gamma_lower': hypgeom_wrappers.acb_hypgeom_gamma_lower_mode(sc, zc, impl='rigorous', prec_bits=53, regularized=True),
    'metadata': api.get_public_function_metadata('hypgeom.arb_hypgeom_1f1'),
}
display(direct_results)

## Production Pattern

Hypergeom production usage should keep family ownership explicit: use the compiled point binder for repeated point traffic, keep `pad_to` fixed, and use the family-owned mode batch wrappers or interval binder for tighter interval traffic.

In [ ]:
onef1_service = api.bind_point_batch_jit('hypgeom.arb_hypgeom_1f1', dtype='float64', pad_to=8)
onef1_interval_service = api.bind_interval_batch('hypgeom.arb_hypgeom_1f1', mode='rigorous', dtype='float64', pad_to=8, prec_bits=53)
service_results = {
    'point_reuse': onef1_service(a, b, z),
    'interval_reuse': onef1_interval_service(a_iv, b_iv, z_iv),
    'family_mode_wrapper': hypgeom_wrappers.arb_hypgeom_gamma_upper_batch_mode_padded(s, x, pad_to=4, impl='adaptive', prec_bits=53, regularized=True),
}
display(service_results)

## Extending Benchmarks

To extend hypergeom benchmarks, add the next family row to `special_function_hardening_benchmark.py` for cross-family diagnostics or to `benchmark_hypgeom.py` / `run_hypgeom_point_kernel_benchmark.py` when the surface needs a dedicated family benchmark.

## Fast JAX Point Pattern

Hypergeom fast JAX should use the family-owned compiled point binder rather than ad hoc `jax.jit` over arbitrary arrays. This section compares the bound padded kernel to the scalar public point path.

In [ ]:
import jax
onef1_fast = api.bind_point_batch_jit('hypgeom.arb_hypgeom_1f1', dtype='float64', pad_to=8)
fast_vals = onef1_fast(a, b, z)
scalar_vals = jax.vmap(lambda aa, bb, zz: api.eval_point('hypgeom.arb_hypgeom_1f1', aa, bb, zz))(a, b, z)
display({'jit_shape': fast_vals.shape, 'jit_matches_scalar_vmap': bool(jnp.allclose(fast_vals, scalar_vals, rtol=1e-6, atol=1e-6))})

## AD Product Pattern

AD should be shown on the real production hypergeom entrypoint in both directions: through the evaluation variable and through a family parameter. This section differentiates the public `hypgeom.arb_hypgeom_1f1` point path over `z` and `a` sweeps and plots both sensitivities.

In [ ]:
import jax
a0 = jnp.float64(1.25)
b0 = jnp.float64(2.25)
z0 = jnp.float64(0.3)
def hypgeom_loss_z(zv):
    return api.eval_point('hypgeom.arb_hypgeom_1f1', a0, b0, zv)
def hypgeom_loss_a(av):
    return api.eval_point('hypgeom.arb_hypgeom_1f1', av, b0, z0)
z_sweep = jnp.linspace(0.1, 0.8, 24, dtype=jnp.float64)
a_sweep = jnp.linspace(0.9, 1.6, 24, dtype=jnp.float64)
primal_z = jax.vmap(hypgeom_loss_z)(z_sweep)
grad_z = jax.vmap(jax.grad(hypgeom_loss_z))(z_sweep)
primal_a = jax.vmap(hypgeom_loss_a)(a_sweep)
grad_a = jax.vmap(jax.grad(hypgeom_loss_a))(a_sweep)
ad_df = pd.DataFrame({'z': [float(v) for v in z_sweep], 'primal_z': [float(v) for v in primal_z], 'grad_z': [float(v) for v in grad_z], 'a': [float(v) for v in a_sweep], 'primal_a': [float(v) for v in primal_a], 'grad_a': [float(v) for v in grad_a]})
display(ad_df.head())
ax = ad_df.plot(x='z', y=['primal_z', 'grad_z'], figsize=(8, 4), title='Hypgeom AD Validation: Argument Direction')
plt.tight_layout()
plt.savefig(EXAMPLE_OUTPUT_ROOT / f'ad_validation_argument_{JAX_MODE}.png', dpi=160, bbox_inches='tight')
plt.show()
ax = ad_df.plot(x='a', y=['primal_a', 'grad_a'], figsize=(8, 4), title='Hypgeom AD Validation: Parameter Direction')
plt.tight_layout()
plt.savefig(EXAMPLE_OUTPUT_ROOT / f'ad_validation_parameter_{JAX_MODE}.png', dpi=160, bbox_inches='tight')
plt.show()

## Validation Summary

Run the current hypergeom engineering, wrapper, and special-function hardening tests.

In [ ]:
tests = run([
    PYTHON, '-m', 'pytest', '-q',
    'tests/test_hypgeom_wrappers_contracts.py',
    'tests/test_hypgeom_engineering.py',
    'tests/test_hypgeom_startup_lazy_loading.py',
    'tests/test_special_function_hardening.py',
], capture=True)
print(tests.stdout)
if tests.stderr:
    print(tests.stderr)
(EXAMPLE_OUTPUT_ROOT / f'pytest_{JAX_MODE}.txt').write_text(tests.stdout + ('\n' + tests.stderr if tests.stderr else ''), encoding='utf-8')

## Benchmark Summary

Run the cross-family hardening benchmark and the dedicated hypgeom startup probe, then structure the emitted artifacts.

In [ ]:
run([PYTHON, 'benchmarks/special_function_hardening_benchmark.py'], capture=True)
run([PYTHON, 'benchmarks/hypgeom_point_startup_probe.py'], capture=True)
hardening_payload = json.loads((REPO_ROOT / 'benchmarks' / 'results' / 'special_function_hardening_benchmark' / 'special_function_hardening_benchmark.json').read_text(encoding='utf-8'))
startup_payload = json.loads((REPO_ROOT / 'benchmarks' / 'results' / 'hypgeom_point_startup_probe' / 'hypgeom_point_startup_probe.json').read_text(encoding='utf-8'))
bench_df = pd.DataFrame([
    {'metric': 'onef1_point_batch_s', 'value': hardening_payload['hypgeom']['onef1_point_batch_s']},
    {'metric': 'gamma_lower_regularized_batch_s', 'value': hardening_payload['hypgeom']['gamma_lower_regularized_batch_s']},
    {'metric': 'gamma_upper_regularized_batch_s', 'value': hardening_payload['hypgeom']['gamma_upper_regularized_batch_s']},
    {'metric': 'startup_import_s', 'value': startup_payload['import_arbplusjax_api']['seconds']},
    {'metric': 'startup_compile_plus_first_s', 'value': startup_payload['arb_hypgeom_1f1_point_path']['compile_plus_first_point_batch_s']},
    {'metric': 'startup_steady_s', 'value': startup_payload['arb_hypgeom_1f1_point_path']['steady_point_batch_s']},
])
bench_df.to_csv(EXAMPLE_OUTPUT_ROOT / f'hypgeom_summary_{JAX_MODE}.csv', index=False)
display(bench_df)

## Comparison Summary

Use the generated engineering reports as the canonical comparison/status layer for the current hypergeom surface.

In [ ]:
status_text = (REPO_ROOT / 'docs' / 'reports' / 'hypgeom_status.md').read_text(encoding='utf-8')
special_status = (REPO_ROOT / 'docs' / 'reports' / 'special_function_status.md').read_text(encoding='utf-8')
comparison_summary = {
    'hypgeom_status_has_1f1': 'arb_hypgeom_1f1 / acb_hypgeom_1f1' in status_text,
    'special_status_has_startup': 'hypgeom_point_startup_probe.json' in special_status,
}
display(comparison_summary)

## Optional Diagnostics

Keep a compact diagnostics summary tied to the startup probe and current hardening report.

In [ ]:
diag_rows = bench_df.copy()
diag_rows.to_csv(EXAMPLE_OUTPUT_ROOT / f'hypgeom_diagnostics_{JAX_MODE}.csv', index=False)
top = bench_df.head(6)
ax = top.plot(x='metric', y='value', kind='barh', figsize=(10, 4), color='#50717d', legend=False, title='Hypgeom Benchmark / Startup Summary')
ax.set_xlabel('value')
plt.tight_layout()
plt.savefig(EXAMPLE_OUTPUT_ROOT / f'hypgeom_summary_{JAX_MODE}.png', dpi=160, bbox_inches='tight')
plt.show()
summary_lines = [
    f'# Example Hypgeom Family Surface Summary ({JAX_MODE})',
    '',
    f'- backend: `{runtime_payload["platform"]}`',
    f'- benchmark_rows: `{len(bench_df)}`',
    f'- diagnostics_rows: `{len(diag_rows)}`',
    '',
    '## Key Metrics',
    '',
]
for row in top.to_dict(orient='records'):
    summary_lines.append(f"- `{row['metric']}`: `{row['value']}`")
(EXAMPLE_OUTPUT_ROOT / f'summary_{JAX_MODE}.md').write_text('\n'.join(summary_lines) + '\n', encoding='utf-8')
display('\n'.join(summary_lines[:14]))